<a href="https://colab.research.google.com/github/myuyen0304/retail-sales-forecasting-research/blob/main/notebooks/01_colab_cpu_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pilot CPU — Retail Sales Forecasting Research

**Chỉ chạy trên Colab hosted CPU với Python 3.13.** 9 ứng viên × G/COGS × H=30/548, origin 31/12/2019. Đây là development pilot, không chấm final test và chưa phải toàn bộ benchmark.

Bản notebook tự chứa module/config từ repo nghiên cứu. Không cần push GitHub hoặc cài gì trên máy cá nhân. Dữ liệu và output lưu trong Drive riêng. Chưa có kết quả huấn luyện đi kèm notebook này.

**Giới hạn quan trọng:** pilot chỉ có sáu baseline, Ridge, LightGBM và tên lịch sử log_sarimax. Tên log_sarimax chưa chuẩn vì code không truyền seasonal_order; nếu model này failed, bảng chỉ lọc success sẽ che nó. Các model RF/Extra Trees/XGBoost/CatBoost/ARIMA/SARIMA/RNN/LSTM nằm ở notebook 02 và 03 riêng.


In [ ]:
import os, sys
from pathlib import Path
if not os.getenv("COLAB_RELEASE_TAG") or not Path("/content").is_dir():
    raise RuntimeError("Hãy chọn Colab hosted runtime; không chạy local.")
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f"Notebook yêu cầu Python 3.13 trên Colab; runtime hiện tại: {sys.version.split()[0]}.")
print("Cloud runtime: Colab, Python", sys.version.split()[0], "; chưa train.")


## 1. Cài môi trường trên VM Colab

Dùng runtime Colab hosted CPU mặc định với Python 3.13. Nếu đang dùng runtime cũ, chuyển về **Latest runtime** trước khi chạy. Cell tiếp theo kiểm phụ thuộc bằng dry-run; cell sau đó chỉ cài wheel, không biên dịch gói hoặc train. Nếu Colab yêu cầu **Restart session** sau khi cài, hãy restart và chạy lại từ cell đầu để kernel nạp đúng phiên bản mới.


In [ ]:
import subprocess, sys
packages = ["numpy==2.1.3", "pandas==2.2.3", "scikit-learn==1.6.1", "lightgbm==4.6.0", "statsmodels==0.14.4", "matplotlib==3.9.4", "pyarrow==18.1.0", "joblib==1.4.2", "databricks-sql-connector==4.6.0"]
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--dry-run", "--only-binary=:all:", *packages],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print("\n".join(result.stdout.splitlines()[-30:]))
result.check_returncode()


In [ ]:
import subprocess, sys
packages = ["numpy==2.1.3", "pandas==2.2.3", "scikit-learn==1.6.1", "lightgbm==4.6.0", "statsmodels==0.14.4", "matplotlib==3.9.4", "pyarrow==18.1.0", "joblib==1.4.2", "databricks-sql-connector==4.6.0"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *packages])
from importlib.metadata import version
for package in packages:
    name, expected = package.split("==")
    actual = version(name)
    if actual != expected:
        raise RuntimeError(f"{name}: cần {expected}, hiện có {actual}.")
subprocess.check_call([sys.executable, "-c", "import numpy, pandas, scipy, sklearn, lightgbm, statsmodels, matplotlib, pyarrow; from databricks import sql; print('Import smoke: OK')"])
print("Đã cài và kiểm tra thư viện trên Python", sys.version.split()[0])


## 2. Tạo bộ chạy và kiểm tra contract trên cloud

Các test ở đây dùng số nhỏ để kiểm metric/split/feature và không fit model. Bản thân các test chạy trên VM Colab.


In [ ]:
import json, os
from pathlib import Path
project = Path("/content/retail-sales-forecasting-research")
project.mkdir(exist_ok=True)
embedded_files = json.loads("{\"requirements-cloud.txt\":\"# -*- coding: utf-8 -*-\\n# Môi trường pilot CPU trên Python 3.13 của Colab hosted runtime.\\nnumpy==2.1.3\\npandas==2.2.3\\nscikit-learn==1.6.1\\nlightgbm==4.6.0\\nstatsmodels==0.14.4\\nmatplotlib==3.9.4\\npyarrow==18.1.0\\njoblib==1.4.2\\ndatabricks-sql-connector==4.6.0\\n\",\"configs/pilot_cpu.json\":\"{\\n  \\\"protocol_id\\\": \\\"pilot-cpu-v1\\\",\\n  \\\"seed\\\": 42,\\n  \\\"origin\\\": \\\"2019-12-31\\\",\\n  \\\"horizons\\\": [\\n    30,\\n    548\\n  ],\\n  \\\"models\\\": [\\n    \\\"mean\\\",\\n    \\\"naive\\\",\\n    \\\"drift\\\",\\n    \\\"snaive7\\\",\\n    \\\"snaive_year\\\",\\n    \\\"snaive_two_year\\\",\\n    \\\"ridge\\\",\\n    \\\"lightgbm\\\",\\n    \\\"log_sarimax\\\"\\n  ],\\n  \\\"targets\\\": [\\n    \\\"G\\\",\\n    \\\"COGS\\\"\\n  ],\\n  \\\"lag_days\\\": [\\n    1,\\n    2,\\n    3,\\n    7,\\n    14,\\n    28,\\n    30,\\n    60,\\n    90,\\n    180,\\n    365,\\n    730\\n  ],\\n  \\\"rolling_days\\\": [\\n    7,\\n    28,\\n    90,\\n    365\\n  ],\\n  \\\"ridge\\\": {\\n    \\\"alpha\\\": 10\\n  },\\n  \\\"lightgbm\\\": {\\n    \\\"n_estimators\\\": 200,\\n    \\\"num_leaves\\\": 15,\\n    \\\"learning_rate\\\": 0.05,\\n    \\\"min_child_samples\\\": 30,\\n    \\\"verbosity\\\": -1,\\n    \\\"n_jobs\\\": 2,\\n    \\\"deterministic\\\": true,\\n    \\\"force_col_wise\\\": true\\n  },\\n  \\\"log_sarimax\\\": {\\n    \\\"order\\\": [\\n      1,\\n      0,\\n      1\\n    ],\\n    \\\"trend\\\": \\\"c\\\",\\n    \\\"maxiter\\\": 100\\n  },\\n  \\\"fit_note\\\": \\\"Pilot CPU F5; chưa tuning; chưa chấm final test; không phải benchmark đủ 40 model.\\\"\\n}\\n\",\"cloud_pilot.py\":\"\\\"\\\"\\\"Pilot CPU chỉ chạy trên Colab/Databricks; không dùng code hệ thống chính.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport time\\nimport warnings\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nDEVELOPMENT_END = pd.Timestamp(\\\"2021-07-01\\\")\\n\\n\\ndef cloud_runtime():\\n    # Guard chống chạy nhầm; đây không phải cơ chế chứng thực hạ tầng.\\n    if os.getenv(\\\"COLAB_RELEASE_TAG\\\") and Path(\\\"/content\\\").is_dir():\\n        return \\\"colab\\\"\\n    if os.getenv(\\\"DATABRICKS_RUNTIME_VERSION\\\"):\\n        return \\\"databricks\\\"\\n    raise RuntimeError(\\\"Không train local. Chạy trên runtime Colab hoặc Databricks.\\\")\\n\\n\\ndef sha256(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_suffix(path.suffix + \\\".tmp\\\")\\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,\\n                                    allow_nan=False), encoding=\\\"utf-8\\\")\\n    temporary.replace(path)\\n\\n\\ndef validate_data(frame):\\n    required = {\\\"sale_date\\\", \\\"g\\\", \\\"cogs\\\"}\\n    if not required.issubset(frame.columns):\\n        raise ValueError(\\\"CSV phải có sale_date,g,cogs từ rpt_forecast_actual.\\\")\\n    data = frame[list(sorted(required))].copy()\\n    data[\\\"sale_date\\\"] = pd.to_datetime(data[\\\"sale_date\\\"], errors=\\\"raise\\\")\\n    dates = data[\\\"sale_date\\\"]\\n    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():\\n        raise ValueError(\\\"sale_date phải là ngày không có timezone/giờ.\\\")\\n    data = data.sort_values(\\\"sale_date\\\").reset_index(drop=True)\\n    expected = pd.date_range(\\\"2012-07-04\\\", \\\"2022-12-31\\\", freq=\\\"D\\\")\\n    if not pd.DatetimeIndex(data[\\\"sale_date\\\"]).equals(expected):\\n        raise ValueError(\\\"Lịch actual không khớp 3833 ngày đã đăng ký; kiểm lại snapshot.\\\")\\n    for name in (\\\"g\\\", \\\"cogs\\\"):\\n        data[name] = pd.to_numeric(data[name], errors=\\\"raise\\\")\\n        if not np.isfinite(data[name]).all() or (data[name] < 0).any():\\n            raise ValueError(\\\"Target thiếu, âm hoặc không hữu hạn; cần điều tra dữ liệu.\\\")\\n    return data\\n\\n\\ndef load_snapshot(csv_path, manifest_path):\\n    manifest = json.loads(Path(manifest_path).read_text(encoding=\\\"utf-8\\\"))\\n    for key in (\\\"snapshot_id\\\", \\\"release_id\\\", \\\"source_relation\\\", \\\"exported_at\\\",\\n                \\\"sha256\\\", \\\"actual_only\\\"):\\n        if key not in manifest:\\n            raise ValueError(\\\"Manifest thiếu \\\" + key)\\n    if manifest[\\\"actual_only\\\"] is not True:\\n        raise ValueError(\\\"Manifest chưa xác nhận actual_only.\\\")\\n    relation_parts = manifest[\\\"source_relation\\\"].split(\\\".\\\")\\n    if (len(relation_parts) != 3 or relation_parts[-1] != \\\"rpt_forecast_actual\\\"\\n            or relation_parts[-2] != \\\"r_\\\" + manifest[\\\"release_id\\\"] + \\\"_reporting\\\"):\\n        raise ValueError(\\\"Manifest phải chỉ đúng rpt_forecast_actual của release.\\\")\\n    if sha256(csv_path) != manifest[\\\"sha256\\\"]:\\n        raise ValueError(\\\"Checksum CSV khác manifest.\\\")\\n    return validate_data(pd.read_csv(csv_path)), manifest\\n\\n\\ndef score(y, prediction, train):\\n    y, prediction, train = [np.asarray(v, dtype=float) for v in (y, prediction, train)]\\n    if y.shape != prediction.shape or y.size == 0:\\n        raise ValueError(\\\"Prediction và actual phải cùng shape, không rỗng.\\\")\\n    if not all(np.isfinite(v).all() for v in (y, prediction, train)):\\n        raise ValueError(\\\"Metric không nhận NaN/Inf.\\\")\\n    error = prediction - y\\n    mae = float(np.abs(error).mean())\\n    rse_den = float(np.square(y - y.mean()).sum())\\n    mase_den = float(np.abs(train[365:] - train[:-365]).mean()) if len(train) > 365 else 0\\n    wape_den = float(np.abs(y).sum())\\n    nonzero = y != 0\\n    return {\\n        \\\"n\\\": len(y), \\\"mae\\\": mae, \\\"rmse\\\": float(np.sqrt(np.square(error).mean())),\\n        \\\"rse\\\": float(np.square(error).sum() / rse_den) if rse_den else None,\\n        \\\"mape\\\": float(100 * np.mean(np.abs(error / y))) if nonzero.all() else None,\\n        \\\"mape_nonzero\\\": float(100 * np.mean(np.abs(error[nonzero] / y[nonzero]))) if nonzero.any() else None,\\n        \\\"mape_coverage\\\": float(nonzero.mean()),\\n        \\\"mase\\\": mae / mase_den if mase_den else None,\\n        \\\"wape\\\": float(100 * np.abs(error).sum() / wape_den) if wape_den else None,\\n        \\\"bias\\\": float(error.mean()), \\\"total_error\\\": float(error.sum()),\\n    }\\n\\n\\ndef calendar(dates):\\n    dates = pd.DatetimeIndex(dates)\\n    result = pd.DataFrame(index=dates)\\n    # Mã hóa cố định, không học từ validation.\\n    for month in range(2, 13):\\n        result[\\\"month_\\\" + str(month)] = (dates.month == month).astype(float)\\n    result[\\\"day\\\"] = dates.day / 31.0\\n    result[\\\"days_in_month\\\"] = dates.days_in_month / 31.0\\n    result[\\\"days_to_end\\\"] = (dates.days_in_month - dates.day) / 31.0\\n    phase = 2 * np.pi * (dates.dayofyear - 1) / np.where(dates.is_leap_year, 366, 365)\\n    for harmonic in (1, 2, 3):\\n        result[\\\"sin_\\\" + str(harmonic)] = np.sin(harmonic * phase)\\n        result[\\\"cos_\\\" + str(harmonic)] = np.cos(harmonic * phase)\\n    return result\\n\\n\\ndef history_row(values, config):\\n    row = [values[-lag] for lag in config[\\\"lag_days\\\"]]\\n    for window in config[\\\"rolling_days\\\"]:\\n        row.extend([np.mean(values[-window:]), np.std(values[-window:], ddof=0)])\\n    return row\\n\\n\\ndef supervised(train, config):\\n    first = max(config[\\\"lag_days\\\"] + config[\\\"rolling_days\\\"])\\n    values = train.to_numpy(dtype=float)\\n    if len(values) <= first:\\n        raise ValueError(\\\"Không đủ train cho lag đã đăng ký.\\\")\\n    cal = calendar(train.index).to_numpy()\\n    rows = [history_row(values[:i], config) + list(cal[i]) for i in range(first, len(values))]\\n    return np.asarray(rows), values[first:], first\\n\\n\\ndef baseline(train, future, model):\\n    values = train.to_numpy(dtype=float)\\n    if model == \\\"mean\\\":\\n        return np.repeat(values.mean(), len(future))\\n    if model == \\\"naive\\\":\\n        return np.repeat(values[-1], len(future))\\n    if model == \\\"drift\\\":\\n        return values[-1] + np.arange(1, len(future) + 1) * (values[-1] - values[0]) / (len(values) - 1)\\n    known = train.to_dict()\\n    predictions = []\\n    for date in future:\\n        if model == \\\"snaive7\\\":\\n            reference = date - pd.Timedelta(days=7)\\n        else:\\n            years = 1 if model == \\\"snaive_year\\\" else 2\\n            reference = date - pd.DateOffset(years=years)\\n        value = float(known[reference])\\n        predictions.append(value)\\n        known[date] = max(value, 0.0)\\n    return np.asarray(predictions)\\n\\n\\ndef fit_predict(train, future, model_name, config):\\n    cloud_runtime()  # Kiểm trước mọi đường fit.\\n    start = time.perf_counter()\\n    fitted = None\\n    fitted_dates = None\\n    metadata = {\\\"input_set\\\": \\\"F0\\\", \\\"strategy\\\": \\\"native\\\", \\\"point\\\": \\\"point\\\"}\\n    estimator = None\\n    baseline_names = {\\\"mean\\\", \\\"naive\\\", \\\"drift\\\", \\\"snaive7\\\", \\\"snaive_year\\\", \\\"snaive_two_year\\\"}\\n    if model_name in baseline_names:\\n        fit_seconds = time.perf_counter() - start\\n        pred_start = time.perf_counter()\\n        prediction = baseline(train, future, model_name)\\n        estimator = {\\\"method\\\": model_name, \\\"history\\\": train.to_dict()}\\n    elif model_name in {\\\"ridge\\\", \\\"lightgbm\\\"}:\\n        from sklearn.linear_model import Ridge\\n        from sklearn.pipeline import make_pipeline\\n        from sklearn.preprocessing import StandardScaler\\n        features, labels, first = supervised(train, config)\\n        feature_seconds = time.perf_counter() - start\\n        start = time.perf_counter()\\n        if model_name == \\\"ridge\\\":\\n            estimator = make_pipeline(StandardScaler(), Ridge(**config[\\\"ridge\\\"]))\\n        else:\\n            from lightgbm import LGBMRegressor\\n            estimator = LGBMRegressor(random_state=config[\\\"seed\\\"], **config[\\\"lightgbm\\\"])\\n        estimator.fit(features, labels)\\n        fit_seconds = time.perf_counter() - start\\n        fitted = estimator.predict(features)\\n        fitted_dates = train.index[first:]\\n        history = list(train.to_numpy(dtype=float))\\n        cal = calendar(future).to_numpy()\\n        pred_start = time.perf_counter()\\n        prediction = []\\n        for i in range(len(future)):\\n            row = np.asarray([history_row(history, config) + list(cal[i])])\\n            value = float(estimator.predict(row)[0])\\n            prediction.append(value)\\n            history.append(max(value, 0.0))\\n        prediction = np.asarray(prediction)\\n        metadata.update(input_set=\\\"F1\\\", strategy=\\\"recursive\\\", feature_seconds=feature_seconds)\\n    elif model_name == \\\"log_sarimax\\\":\\n        from statsmodels.tsa.statespace.sarimax import SARIMAX\\n        options = config[\\\"log_sarimax\\\"]\\n        with warnings.catch_warnings(record=True) as observed:\\n            warnings.simplefilter(\\\"always\\\")\\n            estimator = SARIMAX(np.log1p(train.to_numpy()), exog=calendar(train.index).to_numpy(),\\n                                order=tuple(options[\\\"order\\\"]), trend=options[\\\"trend\\\"],\\n                                enforce_stationarity=True, enforce_invertibility=True).fit(\\n                                    disp=False, maxiter=options[\\\"maxiter\\\"])\\n        metadata[\\\"warnings\\\"] = [str(w.message) for w in observed]\\n        if not estimator.mle_retvals.get(\\\"converged\\\", False):\\n            raise RuntimeError(\\\"SARIMAX không hội tụ; ghi failed, không nhận score.\\\")\\n        fit_seconds = time.perf_counter() - start\\n        fitted = np.expm1(np.asarray(estimator.fittedvalues))\\n        fitted_dates = train.index\\n        pred_start = time.perf_counter()\\n        prediction = np.expm1(np.asarray(estimator.forecast(len(future), exog=calendar(future).to_numpy())))\\n        metadata.update(input_set=\\\"F1\\\", point=\\\"inverse_log_location\\\", strategy=\\\"native\\\")\\n    else:\\n        raise ValueError(\\\"Model chưa triển khai: \\\" + model_name)\\n    pred_seconds = time.perf_counter() - pred_start\\n    if not np.isfinite(prediction).all():\\n        raise ValueError(\\\"Prediction không hữu hạn.\\\")\\n    return prediction, fitted, fitted_dates, estimator, {\\n        **metadata, \\\"fit_seconds\\\": fit_seconds, \\\"predict_seconds\\\": pred_seconds,\\n    }\\n\\n\\ndef make_figure(train, actual, prediction, fitted, fitted_dates, path, title):\\n    import matplotlib\\n    matplotlib.use(\\\"Agg\\\")\\n    import matplotlib.pyplot as plt\\n    fig, axes = plt.subplots(2, 1, figsize=(12, 7))\\n    axes[0].plot(train.index, train.values, label=\\\"Actual train\\\", linewidth=0.8)\\n    if fitted is not None:\\n        axes[0].plot(fitted_dates, np.maximum(fitted, 0), label=\\\"Fitted (in-sample)\\\", alpha=0.7, linewidth=0.7)\\n    axes[0].set_title(\\\"Train — chỉ dùng chẩn đoán\\\")\\n    axes[1].plot(actual.index, actual.values, label=\\\"Actual validation\\\")\\n    axes[1].plot(actual.index, prediction, label=\\\"Forecast từ origin cố định\\\")\\n    axes[1].set_title(title + \\\" — development F5, không phải final test\\\")\\n    for axis in axes:\\n        axis.set_ylabel(\\\"Giá trị tiền theo nguồn\\\")\\n        axis.legend()\\n    fig.tight_layout()\\n    fig.savefig(path, dpi=180)\\n    plt.close(fig)\\n\\n\\ndef run(csv_path, manifest_path, config_path, output):\\n    runtime = cloud_runtime()\\n    data, source = load_snapshot(csv_path, manifest_path)\\n    config = json.loads(Path(config_path).read_text(encoding=\\\"utf-8\\\"))\\n    origin = pd.Timestamp(config[\\\"origin\\\"])\\n    if origin != pd.Timestamp(\\\"2019-12-31\\\") or config[\\\"horizons\\\"] != [30, 548]:\\n        raise ValueError(\\\"Pilot chỉ chạy F5, H=30/548; không mở final test.\\\")\\n    if config[\\\"targets\\\"] != [\\\"G\\\", \\\"COGS\\\"] or len(set(config[\\\"models\\\"])) != len(config[\\\"models\\\"]):\\n        raise ValueError(\\\"Config target/model không hợp lệ.\\\")\\n    if origin + pd.Timedelta(days=max(config[\\\"horizons\\\"])) > DEVELOPMENT_END:\\n        raise ValueError(\\\"Validation vượt development.\\\")\\n    packages = {}\\n    for name in (\\\"numpy\\\", \\\"pandas\\\", \\\"scikit-learn\\\", \\\"lightgbm\\\", \\\"statsmodels\\\", \\\"matplotlib\\\", \\\"joblib\\\"):\\n        packages[name] = importlib.metadata.version(name)\\n    identity = {\\n        \\\"snapshot_sha256\\\": source[\\\"sha256\\\"], \\\"config_sha256\\\": sha256(config_path),\\n        \\\"code_sha256\\\": sha256(__file__), \\\"runtime\\\": runtime,\\n        \\\"python\\\": platform.python_version(), \\\"packages\\\": packages,\\n        \\\"machine\\\": platform.machine(), \\\"cpu_count\\\": os.cpu_count(),\\n    }\\n    run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]\\n    root = Path(output) / run_id\\n    root.mkdir(parents=True, exist_ok=True)\\n    run_manifest = {\\\"run_id\\\": run_id, \\\"source\\\": source, \\\"config\\\": config, **identity}\\n    write_json(root / \\\"manifest.json\\\", run_manifest)\\n    actual_data = data.set_index(\\\"sale_date\\\")\\n    import joblib\\n    results = []\\n    for target in config[\\\"targets\\\"]:\\n        series = actual_data[\\\"g\\\" if target == \\\"G\\\" else \\\"cogs\\\"]\\n        train = series.loc[:origin].copy()\\n        for horizon in config[\\\"horizons\\\"]:\\n            dates = pd.date_range(origin + pd.Timedelta(days=1), periods=horizon)\\n            actual = series.reindex(dates)\\n            for model in config[\\\"models\\\"]:\\n                task = root / (target + \\\"_\\\" + str(horizon) + \\\"_\\\" + model)\\n                task.mkdir(exist_ok=True)\\n                status_path = task / \\\"status.json\\\"\\n                if status_path.exists():\\n                    old = json.loads(status_path.read_text(encoding=\\\"utf-8\\\"))\\n                    hashes = old.get(\\\"artifact_sha256\\\", {})\\n                    if old.get(\\\"status\\\") == \\\"success\\\" and hashes and all(\\n                            (task / name).exists() and sha256(task / name) == digest\\n                            for name, digest in hashes.items()):\\n                        results.append(old)\\n                        continue\\n                record = {\\n                    \\\"run_id\\\": run_id, \\\"snapshot_id\\\": source[\\\"snapshot_id\\\"], \\\"model_id\\\": model,\\n                    \\\"target\\\": target, \\\"origin\\\": str(origin.date()), \\\"horizon\\\": horizon,\\n                    \\\"fold_id\\\": \\\"F5\\\", \\\"split\\\": \\\"development\\\", \\\"seed\\\": config[\\\"seed\\\"],\\n                }\\n                write_json(status_path, {**record, \\\"status\\\": \\\"running\\\"})\\n                try:\\n                    raw, fitted, fitted_dates, estimator, timing = fit_predict(train, dates, model, config)\\n                    prediction = np.maximum(raw, 0)\\n                    predictions = pd.DataFrame({\\n                        **{key: [value] * horizon for key, value in record.items()},\\n                        \\\"ds\\\": dates, \\\"lead_time\\\": np.arange(1, horizon + 1),\\n                        \\\"y_true\\\": actual.to_numpy(), \\\"y_pred_raw\\\": raw, \\\"y_pred\\\": prediction,\\n                    })\\n                    predictions.to_csv(task / \\\"predictions.csv\\\", index=False)\\n                    joblib.dump(estimator, task / \\\"model.joblib\\\")\\n                    if fitted is not None:\\n                        pd.DataFrame({\\\"ds\\\": fitted_dates, \\\"fitted_raw\\\": fitted}).to_csv(task / \\\"fitted.csv\\\", index=False)\\n                    make_figure(train, actual, prediction, fitted, fitted_dates, task / \\\"plot.png\\\",\\n                                target + \\\" / \\\" + model + \\\" / H=\\\" + str(horizon))\\n                    artifact_names = [\\\"predictions.csv\\\", \\\"model.joblib\\\", \\\"plot.png\\\"]\\n                    if fitted is not None:\\n                        artifact_names.append(\\\"fitted.csv\\\")\\n                    record.update(status=\\\"success\\\", **score(actual.to_numpy(), prediction, train.to_numpy()),\\n                                  **timing, clipped_negative=int((raw < 0).sum()),\\n                                  artifact_sha256={name: sha256(task / name) for name in artifact_names})\\n                except Exception as exc:\\n                    record.update(status=\\\"failed\\\", error_type=type(exc).__name__, error=str(exc))\\n                write_json(status_path, record)\\n                results.append(record)\\n                print(target, horizon, model, record[\\\"status\\\"], flush=True)\\n                pd.DataFrame(results).drop(columns=[\\\"artifact_sha256\\\"], errors=\\\"ignore\\\").to_csv(\\n                    root / \\\"metrics.csv\\\", index=False)\\n    pd.DataFrame(results).drop(columns=[\\\"artifact_sha256\\\"], errors=\\\"ignore\\\").to_csv(root / \\\"metrics.csv\\\", index=False)\\n    write_json(root / \\\"summary.json\\\", {\\\"total\\\": len(results),\\n                                     \\\"success\\\": sum(r[\\\"status\\\"] == \\\"success\\\" for r in results),\\n                                     \\\"failed\\\": sum(r[\\\"status\\\"] != \\\"success\\\" for r in results)})\\n    return str(root)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\\"--csv\\\", required=True)\\n    parser.add_argument(\\\"--manifest\\\", required=True)\\n    parser.add_argument(\\\"--config\\\", default=\\\"configs/pilot_cpu.json\\\")\\n    parser.add_argument(\\\"--output\\\", required=True)\\n    args = parser.parse_args()\\n    print(run(args.csv, args.manifest, args.config, args.output))\\n\",\"export_snapshot.py\":\"\\\"\\\"\\\"Export actual bằng callback SELECT, chỉ thực thi trên cloud.\\\"\\\"\\\"\\nfrom datetime import datetime, timezone\\nfrom decimal import Decimal\\nimport json\\nfrom pathlib import Path\\nimport re\\n\\nfrom cloud_pilot import cloud_runtime, sha256, validate_data, write_json\\n\\n\\ndef quoted(identifier):\\n    if not re.fullmatch(r\\\"[A-Za-z0-9_-]+\\\", identifier):\\n        raise ValueError(\\\"Identifier không hợp lệ.\\\")\\n    return chr(96) + identifier + chr(96)\\n\\n\\ndef export_actual(query, catalog, release_id, output):\\n    cloud_runtime()\\n    if not re.fullmatch(r\\\"[a-f0-9]{24}\\\", release_id):\\n        raise ValueError(\\\"Cần release_id 24 ký tự hex đã chọn.\\\")\\n    catalog_sql = quoted(catalog)\\n    state_sql = \\\"SELECT payload FROM \\\" + catalog_sql + \\\".retail_control.state WHERE name = 'main'\\\"\\n    state_frame = query(state_sql)\\n    if len(state_frame) != 1:\\n        raise ValueError(\\\"Registry phải có đúng một bản ghi main.\\\")\\n    payload = state_frame.iloc[0][\\\"payload\\\"]\\n    state = json.loads(payload) if isinstance(payload, str) else payload\\n    record = state.get(\\\"releases\\\", {}).get(release_id)\\n    schema = \\\"r_\\\" + release_id + \\\"_reporting\\\"\\n    if not record or record.get(\\\"status\\\") != \\\"ready\\\" or record.get(\\\"schema\\\") != schema or record.get(\\\"release_id\\\") != release_id:\\n        raise ValueError(\\\"Release không tồn tại hoặc chưa ready.\\\")\\n    relation = catalog + \\\".\\\" + schema + \\\".rpt_forecast_actual\\\"\\n    sql_relation = catalog_sql + \\\".\\\" + quoted(schema) + \\\".rpt_forecast_actual\\\"\\n    select_sql = \\\"SELECT sale_date, g, cogs FROM \\\" + sql_relation + \\\" ORDER BY sale_date\\\"\\n    frame = query(select_sql)\\n    if len(frame) != 3833:\\n        raise ValueError(\\\"Số dòng khác contract 3833; chưa tạo snapshot.\\\")\\n    audit_sql = (\\\"SELECT COUNT(*) AS row_count, MIN(sale_date) AS first_date, \\\"\\n                 \\\"MAX(sale_date) AS last_date, SUM(g) AS total_g, SUM(cogs) AS total_cogs FROM \\\"\\n                 + sql_relation)\\n    audit = query(audit_sql)\\n    if len(audit) != 1 or int(audit.iloc[0][\\\"row_count\\\"]) != len(frame):\\n        raise ValueError(\\\"Đối soát số dòng thất bại.\\\")\\n    for column in (\\\"g\\\", \\\"cogs\\\"):\\n        total = sum((Decimal(str(value)) for value in frame[column]), Decimal(\\\"0\\\"))\\n        expected = Decimal(str(audit.iloc[0][\\\"total_\\\" + column]))\\n        if abs(total - expected) > Decimal(\\\"0.01\\\"):\\n            raise ValueError(\\\"Đối soát tổng thất bại: \\\" + column)\\n    monthly_sql = (\\\"SELECT date_trunc('MONTH', sale_date) AS month, SUM(g) AS g, SUM(cogs) AS cogs FROM \\\"\\n                   + sql_relation + \\\" GROUP BY 1 ORDER BY 1\\\")\\n    monthly = query(monthly_sql)\\n    data = validate_data(frame)\\n    import pandas as pd\\n    local_monthly = data.assign(month=data[\\\"sale_date\\\"].dt.to_period(\\\"M\\\")).groupby(\\\"month\\\")[[\\\"g\\\", \\\"cogs\\\"]].sum()\\n    monthly[\\\"month\\\"] = pd.to_datetime(monthly[\\\"month\\\"]).dt.to_period(\\\"M\\\")\\n    remote_monthly = monthly.set_index(\\\"month\\\")[[\\\"g\\\", \\\"cogs\\\"]].astype(float)\\n    if not local_monthly.index.equals(remote_monthly.index):\\n        raise ValueError(\\\"Đối soát tập tháng thất bại.\\\")\\n    if ((local_monthly - remote_monthly).abs() > 0.01).any().any():\\n        raise ValueError(\\\"Đối soát tổng tháng thất bại.\\\")\\n    root = Path(output) / release_id\\n    root.mkdir(parents=True, exist_ok=True)\\n    csv_path = root / \\\"actual.csv\\\"\\n    if csv_path.exists() or (root / \\\"manifest.json\\\").exists():\\n        raise FileExistsError(\\\"Snapshot đã có; dùng lại hoặc chọn thư mục mới, không ghi đè.\\\")\\n    data.to_csv(csv_path, index=False, date_format=\\\"%Y-%m-%d\\\")\\n    digest = sha256(csv_path)\\n    manifest = {\\n        \\\"snapshot_id\\\": digest[:20], \\\"sha256\\\": digest, \\\"release_id\\\": release_id,\\n        \\\"source_relation\\\": relation, \\\"actual_only\\\": True,\\n        \\\"exported_at\\\": datetime.now(timezone.utc).isoformat(),\\n        \\\"row_count\\\": len(data), \\\"first_date\\\": str(data.sale_date.min().date()),\\n        \\\"last_date\\\": str(data.sale_date.max().date()),\\n        \\\"total_g\\\": str(audit.iloc[0][\\\"total_g\\\"]), \\\"total_cogs\\\": str(audit.iloc[0][\\\"total_cogs\\\"]),\\n        \\\"monthly_reconciled\\\": True,\\n        \\\"queries\\\": [state_sql, select_sql, audit_sql, monthly_sql],\\n    }\\n    write_json(root / \\\"manifest.json\\\", manifest)\\n    return str(csv_path), str(root / \\\"manifest.json\\\")\\n\",\"tests/test_protocol.py\":\"\\\"\\\"\\\"Kiểm contract/metric/feature; không fit model và không cần dữ liệu thật.\\\"\\\"\\\"\\nimport os\\nimport unittest\\nfrom unittest.mock import patch\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom cloud_pilot import baseline, cloud_runtime, score, supervised, validate_data\\n\\n\\nclass ProtocolTests(unittest.TestCase):\\n    def test_local_runtime_is_blocked(self):\\n        with patch.dict(os.environ, {}, clear=True):\\n            with self.assertRaises(RuntimeError):\\n                cloud_runtime()\\n\\n    def test_metrics_hand_calculation(self):\\n        result = score([2, 4], [1, 4], np.arange(800))\\n        self.assertAlmostEqual(result[\\\"mae\\\"], 0.5)\\n        self.assertAlmostEqual(result[\\\"rse\\\"], 0.5)\\n        self.assertAlmostEqual(result[\\\"mape\\\"], 25.0)\\n        self.assertAlmostEqual(result[\\\"rmse\\\"], np.sqrt(0.5))\\n\\n    def test_undefined_metrics(self):\\n        result = score([0, 0], [1, 0], np.ones(800))\\n        for key in (\\\"mape\\\", \\\"rse\\\", \\\"mase\\\", \\\"wape\\\"):\\n            self.assertIsNone(result[key])\\n        self.assertEqual(result[\\\"mape_coverage\\\"], 0.0)\\n\\n    def test_placeholder_dates_rejected(self):\\n        dates = pd.date_range(\\\"2012-07-04\\\", \\\"2022-12-31\\\")\\n        frame = pd.DataFrame({\\\"sale_date\\\": dates, \\\"g\\\": 1.0, \\\"cogs\\\": 2.0})\\n        self.assertEqual(len(validate_data(frame)), 3833)\\n        frame.loc[len(frame)] = [pd.Timestamp(\\\"2023-01-01\\\"), 1.0, 1.0]\\n        with self.assertRaises(ValueError):\\n            validate_data(frame)\\n\\n    def test_seasonal_calendar_leap_day_and_recursive(self):\\n        dates = pd.date_range(\\\"2017-01-01\\\", \\\"2019-12-31\\\")\\n        train = pd.Series(np.arange(len(dates), dtype=float), index=dates)\\n        result = baseline(train, pd.date_range(\\\"2020-01-01\\\", periods=548), \\\"snaive_year\\\")\\n        self.assertEqual(result[59], train.loc[\\\"2019-02-28\\\"])\\n        self.assertEqual(result[366], train.loc[\\\"2019-01-01\\\"])\\n\\n    def test_features_do_not_read_future(self):\\n        config = {\\\"lag_days\\\": [1, 7], \\\"rolling_days\\\": [7]}\\n        series = pd.Series(np.arange(60, dtype=float), index=pd.date_range(\\\"2020-01-01\\\", periods=60))\\n        train = series.loc[:\\\"2020-01-31\\\"]\\n        before = supervised(train, config)\\n        series.loc[\\\"2020-02-01\\\":] = 99999\\n        after = supervised(series.loc[:\\\"2020-01-31\\\"], config)\\n        np.testing.assert_array_equal(before[0], after[0])\\n        self.assertEqual(before[0][0][0], 6.0)\\n        self.assertEqual(before[1][0], 7.0)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    unittest.main()\\n\"}")
for name, text in embedded_files.items():
    path = project / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
os.chdir(project)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"])


## 3. Chọn thư mục Drive lưu dữ liệu và kết quả

Không dùng thư mục chia sẻ công khai. Mỗi task được lưu ngay, mất phiên có thể chạy lại notebook để resume. Ghi thư viện thực tế vào môi trường kết quả để đối chiếu.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
storage = Path("/content/drive/MyDrive/retail-sales-forecasting-research")
storage.mkdir(parents=True, exist_ok=True)
print("Nơi lưu artifact:", storage)


## 4. Khai báo nguồn actual

Điền catalog và release_id đã công bố (24 ký tự hex). Không lấy release bất kỳ hoặc số trong sample_submission. Không điền token trong code. Nếu chưa biết release, cần xác nhận nguồn DWH trước khi chạy.

Colab miễn phí không có nghĩa SQL warehouse Databricks miễn phí: chỉ đọc endpoint/tài khoản đã được phép dùng và còn quota. Không tạo compute mới trong notebook này.


In [ ]:
CATALOG = ""  # Catalog nguồn đã được chọn
RELEASE_ID = ""  # Release ready của DWH
if not CATALOG or not RELEASE_ID:
    raise ValueError("Điền CATALOG và RELEASE_ID của dữ liệu muốn nghiên cứu.")
snapshot_dir = storage / "snapshots" / RELEASE_ID
csv_path = snapshot_dir / "actual.csv"
manifest_path = snapshot_dir / "manifest.json"


## 5. Export chỉ đọc, hoặc dùng lại snapshot đã lưu

Lần đầu nhập hostname, HTTP path của SQL warehouse và token có quyền SELECT. Token được nhập ẩn bằng getpass, không lưu vào notebook hoặc manifest. Dùng bí mật của tài khoản bạn; không gửi token vào chat.

Nếu snapshot đã có, cell sẽ chỉ xác nhận checksum/contract. Export gồm đối soát số dòng, tổng target và tổng tháng. Lỗi đăng nhập cần xử lý trên Databricks; không bỏ qua kiểm tra bằng dữ liệu mẫu.


In [ ]:
from cloud_pilot import load_snapshot
if csv_path.exists() and manifest_path.exists():
    data, manifest = load_snapshot(csv_path, manifest_path)
    print("Dùng lại snapshot:", manifest["snapshot_id"], "rows:", len(data))
else:
    from databricks import sql
    from getpass import getpass
    import pandas as pd
    from export_snapshot import export_actual
    hostname = input("Databricks server hostname (không có https://): ").strip()
    http_path = input("SQL warehouse HTTP path: ").strip()
    token = getpass("Token chỉ đọc (ẩn, không lưu): ")
    try:
        with sql.connect(server_hostname=hostname, http_path=http_path, access_token=token) as connection:
            def query(statement):
                with connection.cursor() as cursor:
                    cursor.execute(statement)
                    return pd.DataFrame([tuple(row) for row in cursor.fetchall()], columns=[item[0] for item in cursor.description])
            csv_result, manifest_result = export_actual(query, CATALOG, RELEASE_ID, storage / "snapshots")
        csv_path, manifest_path = Path(csv_result), Path(manifest_result)
        print("Export và đối soát hoàn tất:", csv_path)
    finally:
        token = None


## 6. Chạy pilot CPU

Cell này bắt đầu training **trên VM Colab**. Không cần GPU. Các model dùng cấu hình cố định, seed 42; không tuning và không mở final test. SARIMAX không hội tụ được ghi failed. Kết quả partial và lỗi được giữ lại, không chỉ báo model thành công.


In [ ]:
from cloud_pilot import run
result_dir = run(csv_path, manifest_path, project / "configs/pilot_cpu.json", storage / "runs")
print("Kết quả:", result_dir)


## 7. Xem metric và biểu đồ

Đây là điểm development pilot; không gọi là accuracy cuối của paper hoặc tuyên bố winner của 40 model. RSE không căn; MAPE không xác định nếu actual có 0. Fitted train chỉ chẩn đoán.

Xem cả bảng lỗi bên dưới trước khi nhận xét model. Không xem 8 dòng success là đủ 9 ứng viên.


In [ ]:
import json
import pandas as pd
result_root = Path(result_dir)
print(json.loads((result_root / "summary.json").read_text()))
metrics = pd.read_csv(result_root / "metrics.csv")
columns = ["target", "horizon", "model_id", "status", "mae", "mape", "rse", "rmse", "fit_seconds", "predict_seconds"]
display(metrics.reindex(columns=columns).sort_values(["target", "horizon", "mae"]))

failed = metrics[metrics["status"] != "success"]
if len(failed):
    print("Model không thành công; xem nguyên nhân:")
    display(failed[[c for c in ["target", "horizon", "model_id", "error_type", "error"] if c in failed.columns]])
else:
    print("Không có model lỗi trong pilot.")


In [ ]:
from IPython.display import Image, display
for plot in sorted(result_root.glob("*/plot.png")):
    display(Image(filename=str(plot)))


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

ok = metrics[metrics["status"].eq("success")].copy()
ok["mae"] = pd.to_numeric(ok["mae"], errors="coerce")

for target in ("G", "COGS"):
    for horizon in (30, 548):
        rank = ok[
            ok["target"].eq(target) & ok["horizon"].eq(horizon)
        ].sort_values("mae").copy()

        print(f"\n{target} — dự báo {horizon} ngày — pilot F5")
        if rank.empty:
            print("Chưa có model chạy thành công.")
            continue

        display(rank[["model_id", "mae", "mape", "rse",
                      "fit_seconds", "predict_seconds"]].reset_index(drop=True))

        fig, ax = plt.subplots(figsize=(13, 4))
        for i, row in enumerate(rank.head(5).itertuples()):
            file = result_root / f"{target}_{horizon}_{row.model_id}" / "predictions.csv"
            pred = pd.read_csv(file, parse_dates=["ds"])
            if i == 0:
                ax.plot(pred["ds"], pred["y_true"], color="black",
                        linewidth=2, label="Thực tế")
            ax.plot(pred["ds"], pred["y_pred"],
                    label=f"{row.model_id} — MAE {row.mae:,.0f}")

        ax.set_title(f"{target}, H={horizon}: 5 model MAE thấp nhất trên pilot F5")
        ax.set_ylabel("Giá trị theo đơn vị dữ liệu nguồn")
        ax.legend()
        ax.grid(alpha=0.2)
        plt.show()


## 8. Bàn giao kết quả

Kết quả nằm trong Drive, gồm manifest, metrics.csv, summary.json và từng task. Khi quay lại trao đổi, cung cấp summary/metrics và manifest không chứa secret để đối chiếu. Chưa có checkpoint DL trong pilot này. Mất phiên giữa một model thì chạy lại model đó; task thành công có checksum hợp lệ sẽ được bỏ qua.

Bước tiếp: chạy notebook 02 cho nhóm cây/ARIMA/SARIMA trên Colab CPU, notebook 03 cho RNN/LSTM trên Colab GPU, rồi notebook 04 để so sánh cùng snapshot/fold.
